# Histograms

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/L16B06")

## Load training and test datasets

In [ ]:
import xarray as xr

## Open a netCDF file in a xarray dataset
fname = 'data/garachico256.ens.nc'
ds    = xr.open_dataset(fname)
train = ds['tephra_col_mass']

fname = 'data/garachico2048.ens.nc'
ds    = xr.open_dataset(fname)
test  = ds['tephra_col_mass']

## Load a pre-trained VAE

In [ ]:
import torch
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale

## Load weight parameters and some metadata
fname = output_dir / 'model.pt'
checkpoint = torch.load(fname)

## Recreate the model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
model.load_state_dict(checkpoint['model_state_dict'])

## Generate a VAE ensemble

In [ ]:
## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
transform = MinMaxScale(min_value, max_value)

## Generate nens new samples
nens = 5000
z = torch.randn(nens, checkpoint['LATENT_DIM'])
with torch.no_grad():
    new_sample = model.decode(z)
    x = transform.invert(new_sample).squeeze()

## Plot configuration

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 200

conf1 = {
    'bins': 128,
    'density': True,
    'alpha': 0.6,
    'color': 'gold',
}
conf2 = {
    'histtype': 'step',
    'density': True,
}

coords = [[65, 42],[52, 40],[70, 54],[53, 51]]

## Create histograms

In [ ]:
x_train = train.values
x_vae   = x.numpy()
x_test  = test.values

In [ ]:
fig, axs = plt.subplots(ncols=2, 
                        nrows=len(coords),
                        sharex='row', 
                        sharey='row', 
                        figsize=(10,11),
                        #tight_layout=True
                       )

for i, coord in enumerate(coords):
    ix,iy = coord
    _, bins, _ = axs[i,0].hist(x_test[:,iy,ix], label = 'Test dataset', **conf1)
    axs[i,0].hist(x_train[:,iy,ix], 
                  bins = bins, 
                  color='royalblue', 
                  label= 'Train dataset', 
                  **conf2)
    _, bins, _ = axs[i,1].hist(x_test[:,iy,ix], label = 'Test dataset', **conf1)
    axs[i,1].hist(x_vae[:,iy,ix], 
                  bins=bins,
                  color='darkred',
                  label = 'VAE - 5000 samples', 
                  **conf2)

labels = (c for c in 'abcdefghijkl')
for i,ax in enumerate(axs.flat):
    ax.set_title(f'({next(labels)}) P{i//2+1}')

axs[3,0].legend()
axs[3,1].legend()

fig.subplots_adjust(left=0.1, 
                    bottom=0.1,
                    right=0.9,
                    top=0.9,
                    wspace=0.1,
                    hspace=0.4)

fig.text(
    0.5,             # X-position: Centered horizontally (0.5)
    0.04,            # Y-position: Near the bottom (0.04)
    r'Ash column mass [$g/m^2$]', 
    ha='center',     # Horizontal alignment: Center the text
    fontsize=12
)

# 5. Add the common Y-axis label
fig.text(
    0.01,            # X-position: Near the left edge (0.01)
    0.5,             # Y-position: Centered vertically (0.5)
    'Probability density',     
    va='center',     # Vertical alignment: Center the text
    rotation='vertical', # Rotate the text 90 degrees
    fontsize=12
)